# SRQ-FLY Phase 1 — state-matched Exact-FLY control
This notebook first chooses the lower Exact-FLY width **from persistent bytes only**, independently selects its Ridge lambda on nested train-only splits, locks all three selections, and then runs six paired confirmation replicates. The existing immutable P2B result ZIP supplies the same-width Exact FLY, P2B, and Raw-Ridge reference rows. Test sets have already been used in prior studies, so this is a secondary confirmation, not a fresh held-out evaluation. Run cells from top to bottom.

In [ ]:
# Edit path/source values only. Do not edit widths, grids, seeds, or methods.
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
REFERENCE_ARTIFACT = '/content/srq_fly_p2b_final_confirmation.zip'
FEATURE_CACHE_ROOT = '/content/srq_state_matched_features'
SELECTION_WTA_ROOT = '/content/srq_state_matched_selection_wta'
FINAL_WTA_ROOT = '/content/srq_state_matched_final_wta'
SELECTION_ROOT = '/content/srq_state_matched_selection'
OUTPUT_ROOT = '/content/srq_state_matched_results'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_CONFIG_SHA256 = 'c77bc7ae6fc627369623c1969cc6daca8f7274fc57c43ff8e9e75c6c547fe84d'
EXPECTED_RUNNER_SHA256 = '52eed46c879568a6b46eaa19e988335173341e814b8753ee909f17c1054d26be'
EXPECTED_REFERENCE_SHA256 = '14826488b8d82bc306a07e6d4f229cc389a8447150833aefc1de664961a9e85d'

In [ ]:
# Fresh clone, dependencies, GPU, and immutable source identities.
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG = 'configs/srq_fly_state_matched_final.json'
RUNNER = 'tools/srq_fly_state_matched_final.py'
assert sha(CONFIG) == EXPECTED_CONFIG_SHA256, (sha(CONFIG), EXPECTED_CONFIG_SHA256)
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256, (sha(RUNNER), EXPECTED_RUNNER_SHA256)
print('SOURCE CHECK: PASS | commit=', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
# Upload the immutable P2B final ZIP with the Colab Files sidebar before this cell.
reference = Path(REFERENCE_ARTIFACT)
assert reference.is_file(), f'Upload {reference.name} to {reference}.'
assert zipfile.is_zipfile(reference), 'Reference artifact is not a valid ZIP.'
assert sha(reference) == EXPECTED_REFERENCE_SHA256, 'P2B reference SHA-256 mismatch.'
print('P2B REFERENCE: PASS | bytes=',reference.stat().st_size,'| sha256=',sha(reference))

In [ ]:
# Download the exact frozen ViT checkpoint and processed dataset sources.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOTS = {
  'cifar100': kagglehub.dataset_download('zaphat206/cifar-100'),
  'cub200': kagglehub.dataset_download('zaphat206/cub-200-2011'),
  'imagenetr': kagglehub.dataset_download('zaphat206/imagenet-r'),
}
print('checkpoint:',CHECKPOINT_PATH); print(json.dumps(DATASET_ROOTS,indent=2))

In [ ]:
# Audit dataset identity without extracting test features.
CUB_AUDIT = '/content/cub_state_matched_audit.json'
IMAGENETR_AUDIT = '/content/imagenetr_state_matched_audit.json'
cub = subprocess.run([sys.executable,'-u','tools/cub_dataset_audit.py','--root',DATASET_ROOTS['cub200'],'--output',CUB_AUDIT,'--expected-identity-sha256','e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca'])
assert cub.returncode == 0
imagenetr = subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOTS['imagenetr'],'--output',IMAGENETR_AUDIT,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4'])
assert imagenetr.returncode == 2, 'Expected disclosed legacy ImageNet-R overlap status 2.'
print('DATASET IDENTITY CHECK: PASS (ImageNet-R legacy overlap retained)')

In [ ]:
# Extract frozen TRAIN features only; selection refuses any visible test.pt.
protocol = json.loads(Path('configs/srq_fly_selfcontained_final.json').read_text())
Path(FEATURE_CACHE_ROOT).mkdir(parents=True,exist_ok=True)
for key in ('cifar100','cub200','imagenetr'):
    cfg=protocol['datasets'][key]; cache=Path(FEATURE_CACHE_ROOT)/key
    if (cache/'test.pt').exists(): (cache/'test.pt').unlink()
    if not (cache/'train.pt').is_file():
        command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],'--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_{key}','--dataset',cfg['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit','--seed','2025','--num-classes',str(cfg['num_classes']),'--num-tasks',str(cfg['num_tasks']),'--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS),'--device','cuda']
        print('TRAIN EXTRACTION START',key,flush=True); subprocess.run(command,check=True)
    assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
print('TRAIN-ONLY FEATURE CACHES: READY')

In [ ]:
# Correctness, byte-derived width, provenance, and leakage gate.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_srq_fly_state_matched_final.py','tests/test_srq_fly_p2b_final_confirmation.py','tests/test_srq_fly_priority1_ablation.py','tests/test_srq_fly_selfcontained.py'],check=True)
for key in ('cifar100','cub200','imagenetr'): assert not (Path(FEATURE_CACHE_ROOT)/key/'test.pt').exists()
print('STATE-MATCHED CORRECTNESS GATE: PASS')

In [ ]:
# Helper: resumable train-only lambda selection at the byte-matched width.
AUDIT_PATHS={'cifar100':None,'cub200':CUB_AUDIT,'imagenetr':IMAGENETR_AUDIT}
def select_dataset(key):
    command=[sys.executable,'-u',RUNNER,'select','--config',CONFIG,'--dataset-key',key,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--code-cache-root',SELECTION_WTA_ROOT,'--output-root',SELECTION_ROOT,'--device','cuda']
    if AUDIT_PATHS[key]: command += ['--dataset-audit',AUDIT_PATHS[key]]
    print(f'SELECTION START {key}: 12 lambdas x 3 development replicates',flush=True)
    subprocess.run(command,check=True)
    result=json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    print('SELECTED',key,'width=',result['state_match']['width'],'lambda=',result['selected_ridge_lambda'],'outer_AIA=',[round(x['result']['validation_average_accuracy'],4) for x in result['outer_confirmation']])
    return result

In [ ]:
# CIFAR-100 train-only selection. Safe to rerun.
cifar_selection=select_dataset('cifar100')

In [ ]:
# CUB-200-2011 train-only selection. Safe to rerun.
cub_selection=select_dataset('cub200')

In [ ]:
# Legacy ImageNet-R train-only selection. Safe to rerun.
imagenetr_selection=select_dataset('imagenetr')

In [ ]:
# Review and download a small train-only checkpoint before crossing the test boundary.
import pandas as pd
selection_rows=[]
for key in ('cifar100','cub200','imagenetr'):
    p=json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    selection_rows.append({'dataset':key,'width':p['state_match']['width'],'lambda':p['selected_ridge_lambda'],'exact_state_bytes':p['state_match']['exact_fly_state_bytes'],'p2b_target_bytes':p['state_match']['target_p2b_state_bytes'],'state_gap_percent':100*p['state_match']['relative_byte_gap'],'status':p['status']})
display(pd.DataFrame(selection_rows))
assert all(row['status']=='SELECTION_COMPLETE' for row in selection_rows), 'Boundary lambda selected; stop before test.'
checkpoint=shutil.make_archive('/content/srq_state_matched_train_only_checkpoint','zip',root_dir=SELECTION_ROOT)
print('TRAIN-ONLY CHECKPOINT READY:',checkpoint,'sha256=',sha(checkpoint))
from google.colab import files
files.download(checkpoint)

In [ ]:
# Lock selections, byte contracts, reference artifact, source, and Git commit.
dirty=subprocess.check_output(['git','status','--porcelain'],text=True).strip()
assert not dirty,f'Repository changed before lock:\n{dirty}'
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-u',RUNNER,'lock','--config',CONFIG,'--selection-root',SELECTION_ROOT,'--reference-artifact',REFERENCE_ARTIFACT,'--output-root',OUTPUT_ROOT,'--require-clean-git'],check=True)
AUTHORIZATION=str(Path(OUTPUT_ROOT)/'state_matched_authorization.json')
authorization=json.loads(Path(AUTHORIZATION).read_text())
print(json.dumps(authorization['selected_hyperparameters'],indent=2)); print('TEST BOUNDARY: AUTHORIZED')

## Test boundary
Everything above used training data only. From the next cell onward test features are visible. Do not alter any width, lambda, seed, method, or stop based on accuracy. Completed replicate units resume only under the identical authorization context.

In [ ]:
# Materialize or validate TEST features only after authorization.
for key in ('cifar100','cub200','imagenetr'):
    command=[sys.executable,'-u',RUNNER,'extract-test','--config',CONFIG,'--dataset-key',key,'--selection-root',SELECTION_ROOT,'--reference-artifact',REFERENCE_ARTIFACT,'--authorization',AUTHORIZATION,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TEST EXTRACTION START',key,flush=True); subprocess.run(command,check=True)
print('ALL AUTHORIZED TEST FEATURE CACHES READY')

In [ ]:
# Helper: six fixed paired replicate identities; only lower-width Exact FLY is newly evaluated.
def run_dataset(key):
    command=[sys.executable,'-u',RUNNER,'evaluate','--config',CONFIG,'--dataset-key',key,'--selection-root',SELECTION_ROOT,'--reference-artifact',REFERENCE_ARTIFACT,'--authorization',AUTHORIZATION,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--code-cache-root',FINAL_WTA_ROOT,'--output-root',OUTPUT_ROOT,'--device','cuda']
    if AUDIT_PATHS[key]: command += ['--dataset-audit',AUDIT_PATHS[key]]
    print(f'FINAL START {key}: 6 state-matched Exact-FLY replicates',flush=True)
    subprocess.run(command,check=True)

In [ ]:
# CIFAR-100 final state-matched control. Safe to rerun.
run_dataset('cifar100')

In [ ]:
# CUB-200-2011 final state-matched control. Safe to rerun.
run_dataset('cub200')

In [ ]:
# Legacy ImageNet-R final state-matched control. Safe to rerun.
run_dataset('imagenetr')

In [ ]:
# Aggregate all four methods and paired confidence intervals.
subprocess.run([sys.executable,'-u',RUNNER,'summarize','--config',CONFIG,'--reference-artifact',REFERENCE_ARTIFACT,'--output-root',OUTPUT_ROOT],check=True)
summary=json.loads(Path(OUTPUT_ROOT,'state_matched_final_summary.json').read_text())
metrics=pd.read_csv(Path(OUTPUT_ROOT,'state_matched_metrics.csv'))
display(metrics[['dataset','method','final_accuracy_mean','final_accuracy_sample_std','average_incremental_accuracy_mean','average_incremental_accuracy_sample_std','persistent_state_bytes_mean']])
print('Paired P2B - state-matched Exact FLY AIA (pp):'); print(json.dumps(summary['paired_p2b_minus_state_matched_fly_aia'],indent=2))
print('Paired P2B - same-width Exact FLY AIA (pp):'); print(json.dumps(summary['paired_p2b_minus_same_width_exact_fly_aia'],indent=2))
print('STATUS:',summary['status']); print('DISCLOSURE:',summary['prior_test_use_disclosure'])

In [ ]:
# Compact publication figure: AIA and persistent-state Pareto view.
import matplotlib.pyplot as plt
import numpy as np
labels={'exact_fly_10000':'Exact FLY-10000','srq_fly_p2b_10000':'SRQ-FLY P2B','exact_fly_state_matched':'State-matched FLY','raw_ridge':'Raw Ridge'}
colors={'exact_fly_10000':'#4C78A8','srq_fly_p2b_10000':'#F58518','exact_fly_state_matched':'#E45756','raw_ridge':'#54A24B'}
datasets=['cifar100','cub200','imagenetr']; methods=list(labels)
fig,axes=plt.subplots(1,2,figsize=(14,5)); x=np.arange(3); width=.2
indexed=metrics.set_index(['dataset','method'])
for j,method in enumerate(methods):
    data=indexed.loc[[(d,method) for d in datasets]]
    axes[0].bar(x+(j-1.5)*width,data['average_incremental_accuracy_mean'],width,yerr=data['average_incremental_accuracy_sample_std'],label=labels[method],color=colors[method],capsize=2)
    axes[1].scatter(data['persistent_state_bytes_mean']/2**20,data['average_incremental_accuracy_mean'],s=70,label=labels[method],color=colors[method])
    for d,xx,yy in zip(datasets,data['persistent_state_bytes_mean']/2**20,data['average_incremental_accuracy_mean']): axes[1].annotate(d,(xx,yy),fontsize=8)
axes[0].set_xticks(x,datasets); axes[0].set_ylabel('AIA (%)'); axes[0].set_title('Accuracy over six paired replicates'); axes[0].legend(fontsize=8)
axes[1].set_xscale('log'); axes[1].set_xlabel('Persistent learner state (MiB, log)'); axes[1].set_ylabel('AIA (%)'); axes[1].set_title('State–accuracy Pareto view'); axes[1].grid(alpha=.25)
fig.tight_layout(); figure=Path(OUTPUT_ROOT)/'state_matched_final.png'; fig.savefig(figure,dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# Export compact evidence; feature/WTA caches and the large reference ZIP are excluded.
staging=Path('/content/srq_fly_state_matched_final_export')
if staging.exists(): shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copytree(OUTPUT_ROOT,staging/'results')
shutil.copytree(SELECTION_ROOT,staging/'train_only_selection')
for source,name in [(CONFIG,'locked_config.json'),(RUNNER,'locked_runner.py'),(CUB_AUDIT,'cub_dataset_audit.json'),(IMAGENETR_AUDIT,'imagenetr_dataset_audit.json')]: shutil.copy2(source,staging/name)
(staging/'reference_artifact_sha256.txt').write_text(sha(REFERENCE_ARTIFACT)+'  '+Path(REFERENCE_ARTIFACT).name+'\n')
(staging/'repo_commit.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],text=True))
bundle=shutil.make_archive('/content/srq_fly_state_matched_final','zip',root_dir=staging.parent,base_dir=staging.name)
print('BUNDLE:',bundle,'bytes=',Path(bundle).stat().st_size,'sha256=',sha(bundle))
files.download(bundle)